# UK Biobank LOOK: Four-Class CFP-OCT Study

This notebook is the interactive entry point for the participant-level bilateral four-class study. Labels are record-derived clinical phenotypes, not expert image grades. Each sample contains left/right CFP and central OCT from the earliest complete bilateral visit. Baseline selection uses complete modalities only, ImageNet V2 ResNet50 branches, linear fusion, bilateral mean aggregation, unweighted cross-entropy, and one-stage end-to-end fine-tuning. Validation selects configurations; sealed balanced and natural-distribution tests remain unavailable until artifacts are explicitly frozen.


## 1. Configuration

Edit only this cell. Every path has a `project.json` default and an optional explicit override. A one-element list runs one setting; longer lists expand a deterministic grid. `baseline_selection` executes LR/dropout calibration, seven fusion positions, three-seed confirmation, and OCT-only/CFP-only references.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/home/mengh/LOOK/2026_09_03_08_30_00")
DATA_ROOT = None                 # e.g. Path("/data/account/LOOK/run")
DATASET_ROOT = None              # root containing the one physical processed dataset
IMAGE_ROOT = None                # root used by all four bilateral image paths
COHORT_ROOT = None               # balanced/natural/incident cohort manifests
LABELS_CSV = None                # balanced primary cohort
NATURAL_LABELS_CSV = None        # natural-distribution secondary cohort
PREPROCESS_CACHE_ROOT = None      # lossless reusable 224x224 cache
CACHE_ROOT = None
RUNS_ROOT = None

GPU_DEVICES = [0, 1]             # [0] for one GPU; [0, 1] for two-GPU DDP
EXECUTION_MODE = "baseline_selection"  # baseline_selection | validation | freeze | test | dry_run
RESUME = True
RESTART = False
CHECK_ALL_IMAGE_PATHS = False
BOOTSTRAP_ITERATIONS = 2000
SMOKE_LIMIT = None

# Used after baseline selection is approved.
BASELINE_SELECTION_MANIFEST = None
FROZEN_STUDY_MANIFEST = None

# Manual study axes. Formal LOOK should instead load BASELINE_SELECTION_MANIFEST.
BACKBONES = ["resnet50"]
FUSION_POSITIONS = ["feature"]  # input | stem | layer1 | layer2 | layer3 | layer4 | feature
SEEDS = [3407]
FILLING_STRATEGIES = ["normalized_mean"]  # normalized_mean | paired_cgan
CLASSIFIER_PROFILES = [{
    "name": "manual", "epochs": 100, "patience": 15,
    "effective_batch_size": 256, "micro_batch_size": 64, "num_workers": 8,
    "pretrained_lr": 1e-4, "new_layer_lr": 1e-3, "weight_decay": 1e-4,
    "warmup_epochs": 5, "classifier_dropout": 0.0, "label_smoothing": 0.0,
    "training_strategy": "end_to_end_finetuning",
    "sampling_strategy": "natural_without_replacement",
    "loss_name": "cross_entropy", "amp": True,
    "baseline_macro_f1_target": 0.65, "baseline_min_class_f1": 0.45,
}]
GAN_PROFILES = [{
    "name": "primary", "gan_validation_fraction": 0.1,
    "gan_epochs": 100, "gan_patience": 10,
    "gan_effective_batch_size": 448, "gan_batch_size": 112,
    "gan_num_workers": 8, "gan_learning_rate": 2e-4,
    "gan_beta1": 0.5, "gan_lambda_l1": 100.0, "gan_base_channels": 64,
}]
LOOK_PROFILES = [{
    "name": "primary", "enabled": True,
    "evaluate_random_missing": True, "evaluate_missing_baselines": True,
    "missing_patterns": ["oct_missing", "cfp_missing"],
    "missing_ratios": [0.2, 0.4, 0.6, 0.8],
    "correction_nodes": ["all_available"],
    "downsample_factors": [4, 8, 16],
    "latent_dims": [16, 32, 64, 128, 256], "max_pca_rank": 256,
    "alpha_grid": [0.0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0],
    "primary_metric": "macro_f1",
}]


## 2. Resolve runtime

The selected LOOK virtual environment must provide the `look_core` and `MHD_Project` packages. Paths are resolved without changing `sys.path`.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, GPU_DEVICES))
import torch
from look_core.paths import ProjectPaths
from look_core.study_grid import (
    StudyGrid, freeze_study_grid, run_baseline_selection, run_study_grid,
    study_grid_from_baseline_selection,
)

PATHS = ProjectPaths.load(
    project_root=PROJECT_ROOT, data_root=DATA_ROOT, dataset_root=DATASET_ROOT,
    image_root=IMAGE_ROOT, cohort_root=COHORT_ROOT, labels_csv=LABELS_CSV,
    natural_labels_csv=NATURAL_LABELS_CSV,
    preprocess_cache_root=PREPROCESS_CACHE_ROOT, cache_root=CACHE_ROOT,
    runs_root=RUNS_ROOT,
)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
assert GPU_DEVICES and len(GPU_DEVICES) == len(set(GPU_DEVICES))
print(PATHS)
print({"device": str(DEVICE), "gpu_devices": GPU_DEVICES})


## 3. Build the study definition

Formal validation/test grids inherit the reviewed frozen baseline. Manual axes are retained for controlled debugging only.


In [ ]:
if BASELINE_SELECTION_MANIFEST:
    GRID = study_grid_from_baseline_selection(
        Path(BASELINE_SELECTION_MANIFEST), filling_strategies=FILLING_STRATEGIES
    )
else:
    GRID = StudyGrid(
        backbones=BACKBONES, fusion_positions=FUSION_POSITIONS, seeds=SEEDS,
        filling_strategies=FILLING_STRATEGIES,
        classifier_profiles=CLASSIFIER_PROFILES, gan_profiles=GAN_PROFILES,
        look_profiles=LOOK_PROFILES,
    )
print(GRID)


## 4. Execute or resume

Completed valid artifacts are reused. Partial checkpoints resume. A changed scientific parameter creates a new deterministic run ID. Test requires the sealed study manifest and evaluates both balanced and natural distributions without updating the model.


In [ ]:
if EXECUTION_MODE == "baseline_selection":
    RESULT = run_baseline_selection(
        PATHS, DEVICE, execute=True, check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
        gpu_devices=tuple(GPU_DEVICES),
    )
elif EXECUTION_MODE == "freeze":
    RESULT = freeze_study_grid(GRID, PATHS, DEVICE, gpu_devices=tuple(GPU_DEVICES))
else:
    PHASE = "test" if EXECUTION_MODE == "test" else "validation"
    RESULT = run_study_grid(
        GRID, PATHS, DEVICE, execute=EXECUTION_MODE != "dry_run", phase=PHASE,
        frozen_manifest=Path(FROZEN_STUDY_MANIFEST) if FROZEN_STUDY_MANIFEST else None,
        resume=RESUME, restart=RESTART, smoke_limit=SMOKE_LIMIT,
        check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS, gpu_devices=tuple(GPU_DEVICES),
    )
RESULT


## 5. Inspect persisted state

The returned object points to persistent plans and manifests. Long training should normally be launched through the detached Step 19 helper; reopening this notebook is not required for resume.


In [ ]:
print({
    "runs_root": str(PATHS.runs_root),
    "status": RESULT.get("status"),
    "manifest": RESULT.get("manifest_path"),
    "quality_gate_passed": RESULT.get("quality_gate_passed"),
    "quality_audit": RESULT.get("quality_audit"),
})
